#### Task 4: Welch t-test (two-sample, unequal variances) - France vs Germany average order value.

In [2]:
import os
import numpy as np
import pandas as pd
 
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
 
 
def load_data():
    """Load the real OnlineRetail.csv if it is in this folder, otherwise the simulated sample."""
    for name in ("OnlineRetail.csv", "OnlineRetail_sample.csv"):
        path = os.path.join(HERE, name)
        if os.path.exists(path):
            df = pd.read_csv(path, encoding="ISO-8859-1", dtype={"InvoiceNo": str, "StockCode": str})
            print(f"Loaded {name}: {len(df):,} rows" + ("" if name == "OnlineRetail.csv" else "  (SIMULATED STAND-IN DATA, not the real file)"))
            break
    else:
        raise FileNotFoundError("Put OnlineRetail.csv (or run generate_sample_data.py) in the same folder.")
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["LineTotal"] = df["Quantity"] * df["UnitPrice"]
    return df
 
 
def valid_rows(df):
    """Drop cancellations (InvoiceNo starts with C) and rows with non-positive quantity or price."""
    return df[~df["InvoiceNo"].str.startswith("C") & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
 
from scipy import stats
 
df = valid_rows(load_data())
orders = df.groupby("InvoiceNo").agg(order_value=("LineTotal", "sum"), country=("Country", "first"))
france = orders.loc[orders["country"] == "France", "order_value"]
germany = orders.loc[orders["country"] == "Germany", "order_value"]
 
print("\nH0: mean order value (France) = mean order value (Germany)")
print("H1: mean order value (France) != mean order value (Germany)   (two-tailed, alpha = 0.05)\n")
 
# Output laid out like Excel's Data Analysis ToolPak: t-Test Two-Sample Assuming Unequal Variances
t_stat, p_two = stats.ttest_ind(france, germany, equal_var=False)
v1, v2, n1, n2 = france.var(ddof=1), germany.var(ddof=1), len(france), len(germany)
df_welch = (v1 / n1 + v2 / n2) ** 2 / ((v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1))
t_crit = stats.t.ppf(0.975, round(df_welch))
table = pd.DataFrame({"France": [france.mean(), v1, n1], "Germany": [germany.mean(), v2, n2]},
                     index=["Mean", "Variance", "Observations"])
print(table.round(2).to_string(), "\n")
print(f"Degrees of freedom (df)        = {round(df_welch)}")
print(f"t Stat                         = {t_stat:.4f}")
print(f"P(T<=t) two-tail               = {p_two:.4f}")
print(f"t Critical two-tail (0.05)     = {t_crit:.4f}\n")
 
if p_two < 0.05:
    print(f"Conclusion: p = {p_two:.4f} < 0.05 (and |t| > t critical): the difference IS statistically significant.")
else:
    print(f"Conclusion: p = {p_two:.4f} >= 0.05 (and |t| < t critical): the difference is NOT statistically significant.")

Loaded OnlineRetail_sample.csv: 3,989 rows  (SIMULATED STAND-IN DATA, not the real file)

H0: mean order value (France) = mean order value (Germany)
H1: mean order value (France) != mean order value (Germany)   (two-tailed, alpha = 0.05)

                 France     Germany
Mean             176.15      775.89
Variance      109462.56  3556818.53
Observations      34.00       47.00 

Degrees of freedom (df)        = 50
t Stat                         = -2.1352
P(T<=t) two-tail               = 0.0377
t Critical two-tail (0.05)     = 2.0086

Conclusion: p = 0.0377 < 0.05 (and |t| > t critical): the difference IS statistically significant.
